# Alpaca Golden Cross Stock Screener

Run the full screener from a notebook: fetch Alpaca daily bars, find recent golden crosses with volume confirmation, calculate support distance, sort closest-to-support first, and save results.

## 1. Setup

Install the project first from the repository root:

```powershell
pip install -e ".[notebook]"
```

Copy `.env.example` to `.env` and add your Alpaca credentials before running the data cells.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd()
if not (repo_root / "alpaca_screener").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from alpaca.data.enums import DataFeed
from alpaca_screener.alpaca_data import (
    clients_from_environment,
    fetch_daily_bars,
    get_tradable_symbols,
)
from alpaca_screener.strategy import ScreenConfig, screen_bars

load_dotenv(repo_root / ".env")
repo_root

## 2. Configure the screen

Use `SYMBOLS` for a fast watchlist run, or set it to an empty list to screen active tradable US equities. While experimenting, keep `LIMIT_UNIVERSE` small.

In [ ]:
SYMBOLS = ["AAPL", "MSFT", "NVDA", "AMD", "AMZN"]
LIMIT_UNIVERSE = None  # Example: 500 when SYMBOLS = []
TOP = 25
FEED = DataFeed.IEX  # Use DataFeed.SIP only when your Alpaca account has SIP access.

config = ScreenConfig(
    crossover_lookback=5,
    volume_window=20,
    volume_multiplier=1.5,
    support_lookback=90,
    support_pivot_span=3,
    min_price=5.0,
    min_average_volume=100_000.0,
)
config

## 3. Fetch Alpaca data

In [ ]:
data_client, trading_client = clients_from_environment()

if SYMBOLS:
    symbols = [symbol.upper() for symbol in SYMBOLS]
else:
    symbols = get_tradable_symbols(trading_client, LIMIT_UNIVERSE)

clock = trading_client.get_clock()
bars_by_symbol = fetch_daily_bars(
    data_client,
    symbols,
    feed=FEED,
    incomplete_session_date=clock.timestamp.date() if clock.is_open else None,
)

len(symbols), len(bars_by_symbol)

## 4. Run the screener

Results are sorted by `distance_to_support_pct` from closest to furthest.

In [ ]:
results = screen_bars(bars_by_symbol, config).head(TOP)
results

## 5. Save results

In [ ]:
output_dir = repo_root / "outputs"
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "screen-results.csv"
json_path = output_dir / "screen-results.json"

results.to_csv(csv_path, index=False)
results.to_json(json_path, orient="records", indent=2)

csv_path, json_path

## 6. Optional: chart the top match

This chart shows close, 50/200 SMAs, and calculated support for the closest-to-support match.

In [ ]:
import matplotlib.pyplot as plt

if results.empty:
    print("No matches to chart.")
else:
    symbol = results.iloc[0]["symbol"]
    support = results.iloc[0]["support"]
    chart = bars_by_symbol[symbol].sort_index().tail(260).copy()
    chart["sma_50"] = chart["close"].rolling(50).mean()
    chart["sma_200"] = chart["close"].rolling(200).mean()

    ax = chart[["close", "sma_50", "sma_200"]].plot(figsize=(12, 6), title=symbol)
    ax.axhline(support, color="tab:green", linestyle="--", label="support")
    ax.set_ylabel("Price")
    ax.legend()
    plt.show()